In [1]:
# Celda 1 — Configuracion inicial: librerias y parches de compatibilidad
#
# Este notebook (02) corre en un kernel propio, independiente del notebook 01.
# Por eso hay que volver a importar y volver a aplicar los parches: no es
# trabajo repetido en vano, es necesario porque cada notebook tiene su propia
# memoria. Esta celda es rapida, no descarga nada todavia.

import pandas as pd     # pandas: manejo de datos en tablas (DataFrames)
import requests          # requests: peticiones HTTP directas a la API de XM
import time               # time: pausas entre llamadas a la API
from pydataxm.pydataxm import ReadDB   # ReadDB: clase principal de pydataxm (catalogo de metricas)

# --- Los mismos 3 parches del notebook 01 ---
# pydataxm 0.3.17 sigue sin actualizarse para pandas 3.0, asi que estos
# parches siguen siendo necesarios en cualquier notebook que use la libreria.

_date_range_original = pd.date_range
def _date_range_parchado(*args, **kwargs):
    if kwargs.get("freq") == "M":
        kwargs["freq"] = "ME"
    return _date_range_original(*args, **kwargs)
pd.date_range = _date_range_parchado

_to_numeric_original = pd.to_numeric
def _to_numeric_parchado(*args, **kwargs):
    if kwargs.get("errors") == "ignore":
        kwargs["errors"] = "coerce"
    return _to_numeric_original(*args, **kwargs)
pd.to_numeric = _to_numeric_parchado

_to_datetime_original = pd.to_datetime
def _to_datetime_parchado(*args, **kwargs):
    if kwargs.get("errors") == "ignore":
        kwargs["errors"] = "coerce"
    return _to_datetime_original(*args, **kwargs)
pd.to_datetime = _to_datetime_parchado

print("Configuracion lista: librerias importadas y parches aplicados")

Configuracion lista: librerias importadas y parches aplicados


In [2]:
# Celda 2 — Conexion con la API de XM
# ReadDB es la clase principal de pydataxm. El catalogo (get_collections) SI
# funciona bien tal cual; el bug que encontramos solo afecta a request_data().
api = ReadDB()
print("Objeto 'api' creado correctamente")

Objeto 'api' creado correctamente


In [3]:
# Celda 3 — Cargar el dataset de precio de bolsa ya descargado
#
# No volvemos a llamar la API (ya esta guardado en data/ desde el notebook 01).
# Lo cargamos tambien aqui porque, cuando tengamos embalses y las demas
# variables, vamos a CRUZARLAS con el precio en este mismo notebook.

df_precio = pd.read_csv("../data/precio_bolsa_2019_2024.csv", parse_dates=["fecha_hora"])
print(df_precio.shape)
df_precio.head()

(52608, 2)


,fecha_hora,precio_bolsa_cop_kwh
0,2019-01-01 00:00:00,318.4231
1,2019-01-01 01:00:00,318.4231
2,2019-01-01 02:00:00,318.4231
3,2019-01-01 03:00:00,318.4231
4,2019-01-01 04:00:00,308.9231


In [4]:
# Celda 4 — Buscar en el catalogo las metricas relacionadas con embalses/hidrologia
catalogo = api.get_collections()

# Buscamos "embalse" en todas las columnas de texto del catalogo.
mascara_embalse = catalogo.apply(lambda columna: columna.astype(str).str.contains("embalse", case=False, na=False))
print("=== Resultados para 'embalse' ===")
print(catalogo[mascara_embalse.any(axis=1)][["MetricId", "MetricName", "Entity", "Type", "MetricUnits"]])

# XM suele separar "volumen almacenado" de "aportes hidricos" (el agua que entra).
# Buscamos tambien esta segunda palabra clave.
mascara_aporte = catalogo.apply(lambda columna: columna.astype(str).str.contains("aporte", case=False, na=False))
print("\n=== Resultados para 'aporte' ===")
print(catalogo[mascara_aporte.any(axis=1)][["MetricId", "MetricName", "Entity", "Type", "MetricUnits"]])

=== Resultados para 'embalse' ===
             MetricId                               MetricName   Entity  \
120          AporEner             Aportes  Energía por Sistema  Sistema   
123  VoluUtilDiarEner  Volumen Útil diario Energía por Sistema  Sistema   
132  CapaUtilDiarEner       Capacidad Útil Energía por Sistema  Sistema   
139  VoluUtilDiarEner  Volumen Útil diario Energía por Embalse  Embalse   
140  CapaUtilDiarEner       Capacidad Útil Energía por Embalse  Embalse   
141          AporEner                 Aportes  Energía por Rio      Rio   
150  VoluUtilDiarMasa          Volumen Útil diario por Embalse  Embalse   
151          VertMasa         Vertimientos Volumen por Embalse  Embalse   
154          DescMasa                    Descargas por Embalse  Embalse   
155          VertEner         Vertimientos Energía por Embalse  Embalse   
156          VertEner         Vertimientos Energía por Sistema  Sistema   
157  CapaUtilDiarMasa       Capacidad Útil Volumen por Embalse  Em

In [5]:
# Celda 5 — Diagnostico: como viene el JSON crudo de una metrica DIARIA
#
# El precio de bolsa es HourlyEntities (24 valores por dia). Estas metricas de
# embalses son DailyEntities, asi que es probable que la estructura del JSON
# sea distinta (un solo valor por dia, no 24). Lo confirmamos antes de escribir
# la funcion de descarga, para no repetir el error de aplanado que ya tuvimos.

import json

cuerpo_peticion = {
    "MetricId": "VoluUtilDiarEner",
    "StartDate": "2024-01-01",
    "EndDate": "2024-01-03",
    "Entity": "Sistema"
}

# OJO: las metricas DailyEntities usan un endpoint distinto al de precio (/hourly).
respuesta = requests.post("https://servapibi.xm.com.co/daily", json=cuerpo_peticion)
respuesta.raise_for_status()

datos_crudos = respuesta.json()
print(json.dumps(datos_crudos["Items"][0], indent=2, ensure_ascii=False))

{
  "Date": "2024-01-01",
  "DailyEntities": [
    {
      "Id": "Sistema",
      "Value": "12161276400.00000"
    }
  ]
}


In [6]:
# Celda 6 — Funcion propia para descargar metricas DIARIAS (embalses, aportes, etc.)
#
# A diferencia del precio de bolsa (HourlyEntities, 24 valores por dia), estas
# metricas son DailyEntities: un solo valor por dia. Por eso la funcion es mas
# simple que descargar_precio_bolsa, y ademas la hacemos generica (recibe el
# MetricId como parametro) para no repetir codigo con volumen y aportes.

def descargar_metrica_diaria(metric_id, fecha_inicio, fecha_fin, entity="Sistema"):
    """
    Descarga una metrica DIARIA de la API de XM (endpoint /daily).
    Maximo 31 dias por llamado (misma restriccion que el precio horario).

    Parametros:
        metric_id: codigo de la metrica, ej. 'VoluUtilDiarEner' o 'AporEner'
        fecha_inicio, fecha_fin: strings 'YYYY-MM-DD'
        entity: nivel de agregacion, 'Sistema' por defecto (agregado nacional)

    Devuelve:
        DataFrame de pandas con columnas 'fecha' y 'valor'.
    """
    cuerpo_peticion = {
        "MetricId": metric_id,
        "StartDate": fecha_inicio,
        "EndDate": fecha_fin,
        "Entity": entity
    }

    # OJO: endpoint /daily, no /hourly como el precio de bolsa.
    respuesta = requests.post("https://servapibi.xm.com.co/daily", json=cuerpo_peticion)
    respuesta.raise_for_status()

    datos = respuesta.json()

    filas = []
    for dia in datos["Items"]:
        fecha = dia["Date"]  # ej. '2024-01-01'

        # 'DailyEntities' es una lista; para Entity='Sistema' trae un solo elemento.
        valor = float(dia["DailyEntities"][0]["Value"])  # viene como texto, lo pasamos a numero

        filas.append({"fecha": pd.Timestamp(fecha), "valor": valor})

    return pd.DataFrame(filas)


# --- Prueba con el mismo rango de antes, para comparar contra el JSON crudo ---
df_test_volumen = descargar_metrica_diaria("VoluUtilDiarEner", "2024-01-01", "2024-01-03")
print(df_test_volumen.shape)
df_test_volumen

(3, 2)


,fecha,valor
0,2024-01-01,1.216128e+10
1,2024-01-02,1.210052e+10
2,2024-01-03,1.203649e+10


In [7]:
# Celda 7 — Bucle robusto para descargar cualquier metrica DIARIA en bloques
#
# Misma logica que descargar_rango_robusto (precio horario), pero generica:
# recibe el metric_id como parametro, para reutilizarla con volumen, aportes,
# o cualquier otra metrica diaria que necesitemos mas adelante.

def descargar_rango_diario_robusto(metric_id, fecha_inicio_total, fecha_fin_total,
                                     dias_por_bloque=30, max_reintentos=3, entity="Sistema"):
    """
    Descarga una metrica diaria para un rango largo, en bloques, con reintentos.

    Devuelve una tupla: (df_completo, bloques_fallidos)
    """
    fecha_actual = pd.Timestamp(fecha_inicio_total)
    fecha_limite = pd.Timestamp(fecha_fin_total)

    bloques_descargados = []
    bloques_fallidos = []
    numero_bloque = 1

    while fecha_actual <= fecha_limite:
        fin_de_bloque = min(fecha_actual + pd.Timedelta(days=dias_por_bloque - 1), fecha_limite)
        inicio_str = fecha_actual.strftime("%Y-%m-%d")
        fin_str = fin_de_bloque.strftime("%Y-%m-%d")

        exito = False
        for intento in range(1, max_reintentos + 1):
            try:
                df_bloque = descargar_metrica_diaria(metric_id, inicio_str, fin_str, entity)
                bloques_descargados.append(df_bloque)
                print(f"[{metric_id}] Bloque {numero_bloque} OK: {inicio_str} a {fin_str} ({len(df_bloque)} filas)")
                exito = True
                break
            except Exception as error:
                print(f"  [{metric_id}] Bloque {numero_bloque} intento {intento} fallo: {error}")
                time.sleep(intento * 3)

        if not exito:
            bloques_fallidos.append((inicio_str, fin_str))
            print(f"  [{metric_id}] Bloque {numero_bloque} DESCARTADO tras {max_reintentos} intentos")

        fecha_actual = fin_de_bloque + pd.Timedelta(days=1)
        numero_bloque += 1
        time.sleep(1)

    if bloques_descargados:
        df_completo = pd.concat(bloques_descargados, ignore_index=True)
        df_completo = df_completo.sort_values("fecha").drop_duplicates("fecha").reset_index(drop=True)
    else:
        df_completo = pd.DataFrame()

    return df_completo, bloques_fallidos

print("Funcion 'descargar_rango_diario_robusto' definida correctamente")

Funcion 'descargar_rango_diario_robusto' definida correctamente


In [8]:
# Celda 8 — Prueba corta con AMBAS metricas (3 meses), antes de ir a 6 años

df_prueba_volumen, fallos_v = descargar_rango_diario_robusto("VoluUtilDiarEner", "2024-01-01", "2024-03-31")
print("\nVolumen ->", df_prueba_volumen.shape, "| fallidos:", len(fallos_v))

df_prueba_aportes, fallos_a = descargar_rango_diario_robusto("AporEner", "2024-01-01", "2024-03-31")
print("Aportes  ->", df_prueba_aportes.shape, "| fallidos:", len(fallos_a))

[VoluUtilDiarEner] Bloque 1 OK: 2024-01-01 a 2024-01-30 (30 filas)
[VoluUtilDiarEner] Bloque 2 OK: 2024-01-31 a 2024-02-29 (30 filas)
[VoluUtilDiarEner] Bloque 3 OK: 2024-03-01 a 2024-03-30 (30 filas)
[VoluUtilDiarEner] Bloque 4 OK: 2024-03-31 a 2024-03-31 (1 filas)

Volumen -> (91, 2) | fallidos: 0
[AporEner] Bloque 1 OK: 2024-01-01 a 2024-01-30 (30 filas)
[AporEner] Bloque 2 OK: 2024-01-31 a 2024-02-29 (30 filas)
[AporEner] Bloque 3 OK: 2024-03-01 a 2024-03-30 (30 filas)
[AporEner] Bloque 4 OK: 2024-03-31 a 2024-03-31 (1 filas)
Aportes  -> (91, 2) | fallidos: 0


In [9]:
# Celda 9 — Mini-prueba: confirmar que existen datos de embalses desde 2019
df_test_vol_2019 = descargar_metrica_diaria("VoluUtilDiarEner", "2019-01-01", "2019-01-05")
print("Volumen 2019 ->", df_test_vol_2019.shape)
print(df_test_vol_2019)

df_test_apor_2019 = descargar_metrica_diaria("AporEner", "2019-01-01", "2019-01-05")
print("\nAportes 2019 ->", df_test_apor_2019.shape)
print(df_test_apor_2019)

Volumen 2019 -> (5, 2)
       fecha         valor
0 2019-01-01  1.210678e+10
1 2019-01-02  1.203963e+10
2 2019-01-03  1.195639e+10
3 2019-01-04  1.187703e+10
4 2019-01-05  1.180290e+10

Aportes 2019 -> (5, 2)
       fecha       valor
0 2019-01-01  63035900.0
1 2019-01-02  60901300.0
2 2019-01-03  64058000.0
3 2019-01-04  62531300.0
4 2019-01-05  62769100.0


In [10]:
# Celda 10 — Descarga COMPLETA 2019-2024 de volumen util y aportes, y guardado en disco
#
# Dos descargas independientes, cada una en su propio archivo .csv dentro de data/.
# Reutilizamos la funcion generica descargar_rango_diario_robusto que ya validamos.

# --- Volumen util diario de embalses (energia almacenada) ---
df_volumen, fallos_volumen = descargar_rango_diario_robusto("VoluUtilDiarEner", "2019-01-01", "2024-12-31")

print("\n===== RESUMEN VOLUMEN =====")
print("Total de filas:", df_volumen.shape[0])
print("Rango de fechas:", df_volumen["fecha"].min(), "a", df_volumen["fecha"].max())
print("Bloques fallidos:", len(fallos_volumen))

df_volumen.to_csv("../data/volumen_util_embalses_2019_2024.csv", index=False)
print("Guardado en: ../data/volumen_util_embalses_2019_2024.csv")

# --- Aportes hidricos diarios (energia equivalente que entra a los embalses) ---
df_aportes, fallos_aportes = descargar_rango_diario_robusto("AporEner", "2019-01-01", "2024-12-31")

print("\n===== RESUMEN APORTES =====")
print("Total de filas:", df_aportes.shape[0])
print("Rango de fechas:", df_aportes["fecha"].min(), "a", df_aportes["fecha"].max())
print("Bloques fallidos:", len(fallos_aportes))

df_aportes.to_csv("../data/aportes_hidricos_2019_2024.csv", index=False)
print("Guardado en: ../data/aportes_hidricos_2019_2024.csv")

[VoluUtilDiarEner] Bloque 1 OK: 2019-01-01 a 2019-01-30 (30 filas)
[VoluUtilDiarEner] Bloque 2 OK: 2019-01-31 a 2019-03-01 (30 filas)
[VoluUtilDiarEner] Bloque 3 OK: 2019-03-02 a 2019-03-31 (30 filas)
[VoluUtilDiarEner] Bloque 4 OK: 2019-04-01 a 2019-04-30 (30 filas)
[VoluUtilDiarEner] Bloque 5 OK: 2019-05-01 a 2019-05-30 (30 filas)
[VoluUtilDiarEner] Bloque 6 OK: 2019-05-31 a 2019-06-29 (30 filas)
[VoluUtilDiarEner] Bloque 7 OK: 2019-06-30 a 2019-07-29 (30 filas)
[VoluUtilDiarEner] Bloque 8 OK: 2019-07-30 a 2019-08-28 (30 filas)
[VoluUtilDiarEner] Bloque 9 OK: 2019-08-29 a 2019-09-27 (30 filas)
[VoluUtilDiarEner] Bloque 10 OK: 2019-09-28 a 2019-10-27 (30 filas)
[VoluUtilDiarEner] Bloque 11 OK: 2019-10-28 a 2019-11-26 (30 filas)
[VoluUtilDiarEner] Bloque 12 OK: 2019-11-27 a 2019-12-26 (30 filas)
[VoluUtilDiarEner] Bloque 13 OK: 2019-12-27 a 2020-01-25 (30 filas)
[VoluUtilDiarEner] Bloque 14 OK: 2020-01-26 a 2020-02-24 (30 filas)
[VoluUtilDiarEner] Bloque 15 OK: 2020-02-25 a 2020-03-25 

In [11]:
# Celda 11 — Descargar SOLO 2025 y extender los archivos ya guardados
#
# Ya tenemos 2019-2024 completo y guardado. En vez de repetir esa descarga,
# solo pedimos el pedazo que falta (2025) y lo concatenamos con lo que ya existe.

# --- Volumen: descargar solo 2025 ---
df_volumen_2025, fallos_v25 = descargar_rango_diario_robusto("VoluUtilDiarEner", "2025-01-01", "2025-12-31")
print("Volumen 2025 ->", df_volumen_2025.shape, "| fallidos:", len(fallos_v25))

# --- Aportes: descargar solo 2025 ---
df_aportes_2025, fallos_a25 = descargar_rango_diario_robusto("AporEner", "2025-01-01", "2025-12-31")
print("Aportes 2025 ->", df_aportes_2025.shape, "| fallidos:", len(fallos_a25))

[VoluUtilDiarEner] Bloque 1 OK: 2025-01-01 a 2025-01-30 (30 filas)
[VoluUtilDiarEner] Bloque 2 OK: 2025-01-31 a 2025-03-01 (30 filas)
[VoluUtilDiarEner] Bloque 3 OK: 2025-03-02 a 2025-03-31 (30 filas)
[VoluUtilDiarEner] Bloque 4 OK: 2025-04-01 a 2025-04-30 (30 filas)
[VoluUtilDiarEner] Bloque 5 OK: 2025-05-01 a 2025-05-30 (30 filas)
[VoluUtilDiarEner] Bloque 6 OK: 2025-05-31 a 2025-06-29 (30 filas)
[VoluUtilDiarEner] Bloque 7 OK: 2025-06-30 a 2025-07-29 (30 filas)
[VoluUtilDiarEner] Bloque 8 OK: 2025-07-30 a 2025-08-28 (30 filas)
[VoluUtilDiarEner] Bloque 9 OK: 2025-08-29 a 2025-09-27 (30 filas)
[VoluUtilDiarEner] Bloque 10 OK: 2025-09-28 a 2025-10-27 (30 filas)
[VoluUtilDiarEner] Bloque 11 OK: 2025-10-28 a 2025-11-26 (30 filas)
[VoluUtilDiarEner] Bloque 12 OK: 2025-11-27 a 2025-12-26 (30 filas)
[VoluUtilDiarEner] Bloque 13 OK: 2025-12-27 a 2025-12-31 (5 filas)
Volumen 2025 -> (365, 2) | fallidos: 0
[AporEner] Bloque 1 OK: 2025-01-01 a 2025-01-30 (30 filas)
[AporEner] Bloque 2 OK: 2025

In [12]:
# Celda 12 — Unir 2025 con lo ya guardado (2019-2024) y volver a guardar

# Cargamos lo que ya teniamos en disco
df_volumen_2019_2024 = pd.read_csv("../data/volumen_util_embalses_2019_2024.csv", parse_dates=["fecha"])
df_aportes_2019_2024 = pd.read_csv("../data/aportes_hidricos_2019_2024.csv", parse_dates=["fecha"])

# Concatenamos con el pedazo nuevo de 2025, ordenamos y quitamos posibles duplicados
df_volumen_completo = pd.concat([df_volumen_2019_2024, df_volumen_2025], ignore_index=True)
df_volumen_completo = df_volumen_completo.sort_values("fecha").drop_duplicates("fecha").reset_index(drop=True)

df_aportes_completo = pd.concat([df_aportes_2019_2024, df_aportes_2025], ignore_index=True)
df_aportes_completo = df_aportes_completo.sort_values("fecha").drop_duplicates("fecha").reset_index(drop=True)

print("Volumen 2019-2025 ->", df_volumen_completo.shape)
print("Aportes 2019-2025 ->", df_aportes_completo.shape)

# Guardamos con nombres actualizados que reflejan el nuevo rango
df_volumen_completo.to_csv("../data/volumen_util_embalses_2019_2025.csv", index=False)
df_aportes_completo.to_csv("../data/aportes_hidricos_2019_2025.csv", index=False)
print("\nGuardado: volumen_util_embalses_2019_2025.csv y aportes_hidricos_2019_2025.csv")

Volumen 2019-2025 -> (2557, 2)
Aportes 2019-2025 -> (2557, 2)

Guardado: volumen_util_embalses_2019_2025.csv y aportes_hidricos_2019_2025.csv


In [13]:
# Celda 13 — Traer la funcion de descarga horaria (definida en el notebook 01)
# para poder extender tambien el precio de bolsa a 2025 desde aqui.

def descargar_precio_bolsa(fecha_inicio, fecha_fin):
    """Descarga el Precio de Bolsa Nacional (COP/kWh) para un rango de fechas."""
    cuerpo_peticion = {
        "MetricId": "PrecBolsNaci",
        "StartDate": fecha_inicio,
        "EndDate": fecha_fin,
        "Entity": "Sistema"
    }
    respuesta = requests.post("https://servapibi.xm.com.co/hourly", json=cuerpo_peticion)
    respuesta.raise_for_status()
    datos = respuesta.json()

    filas = []
    for dia in datos["Items"]:
        fecha = dia["Date"]
        valores_del_dia = dia["HourlyEntities"][0]["Values"]
        for hora in range(1, 25):
            clave_hora = f"Hour{hora:02d}"
            precio = float(valores_del_dia[clave_hora])
            fecha_hora = pd.Timestamp(fecha) + pd.Timedelta(hours=hora - 1)
            filas.append({"fecha_hora": fecha_hora, "precio_bolsa_cop_kwh": precio})
    return pd.DataFrame(filas)


def descargar_rango_robusto(fecha_inicio_total, fecha_fin_total, dias_por_bloque=30, max_reintentos=3):
    """Descarga precio de bolsa en bloques, con reintentos."""
    fecha_actual = pd.Timestamp(fecha_inicio_total)
    fecha_limite = pd.Timestamp(fecha_fin_total)
    bloques_descargados = []
    bloques_fallidos = []
    numero_bloque = 1

    while fecha_actual <= fecha_limite:
        fin_de_bloque = min(fecha_actual + pd.Timedelta(days=dias_por_bloque - 1), fecha_limite)
        inicio_str = fecha_actual.strftime("%Y-%m-%d")
        fin_str = fin_de_bloque.strftime("%Y-%m-%d")

        exito = False
        for intento in range(1, max_reintentos + 1):
            try:
                df_bloque = descargar_precio_bolsa(inicio_str, fin_str)
                bloques_descargados.append(df_bloque)
                print(f"Bloque {numero_bloque} OK: {inicio_str} a {fin_str} ({len(df_bloque)} filas)")
                exito = True
                break
            except Exception as error:
                print(f"  Bloque {numero_bloque} intento {intento} fallo: {error}")
                time.sleep(intento * 3)

        if not exito:
            bloques_fallidos.append((inicio_str, fin_str))
        fecha_actual = fin_de_bloque + pd.Timedelta(days=1)
        numero_bloque += 1
        time.sleep(1)

    if bloques_descargados:
        df_completo = pd.concat(bloques_descargados, ignore_index=True)
        df_completo = df_completo.sort_values("fecha_hora").drop_duplicates("fecha_hora").reset_index(drop=True)
    else:
        df_completo = pd.DataFrame()
    return df_completo, bloques_fallidos

print("Funciones de precio horario listas")

Funciones de precio horario listas


In [14]:
# Celda 14 — Descargar SOLO 2025 del precio y extender el archivo ya guardado
df_precio_2025, fallos_p25 = descargar_rango_robusto("2025-01-01", "2025-12-31")
print("Precio 2025 ->", df_precio_2025.shape, "| fallidos:", len(fallos_p25))

# Unimos con lo que ya teniamos (2019-2024, del notebook 01)
df_precio_2019_2024 = pd.read_csv("../data/precio_bolsa_2019_2024.csv", parse_dates=["fecha_hora"])
df_precio_completo = pd.concat([df_precio_2019_2024, df_precio_2025], ignore_index=True)
df_precio_completo = df_precio_completo.sort_values("fecha_hora").drop_duplicates("fecha_hora").reset_index(drop=True)

print("Precio 2019-2025 ->", df_precio_completo.shape)

df_precio_completo.to_csv("../data/precio_bolsa_2019_2025.csv", index=False)
print("Guardado: precio_bolsa_2019_2025.csv")

Bloque 1 OK: 2025-01-01 a 2025-01-30 (720 filas)
Bloque 2 OK: 2025-01-31 a 2025-03-01 (720 filas)
Bloque 3 OK: 2025-03-02 a 2025-03-31 (720 filas)
Bloque 4 OK: 2025-04-01 a 2025-04-30 (720 filas)
Bloque 5 OK: 2025-05-01 a 2025-05-30 (720 filas)
Bloque 6 OK: 2025-05-31 a 2025-06-29 (720 filas)
Bloque 7 OK: 2025-06-30 a 2025-07-29 (720 filas)
Bloque 8 OK: 2025-07-30 a 2025-08-28 (720 filas)
Bloque 9 OK: 2025-08-29 a 2025-09-27 (720 filas)
Bloque 10 OK: 2025-09-28 a 2025-10-27 (720 filas)
Bloque 11 OK: 2025-10-28 a 2025-11-26 (720 filas)
Bloque 12 OK: 2025-11-27 a 2025-12-26 (720 filas)
Bloque 13 OK: 2025-12-27 a 2025-12-31 (120 filas)
Precio 2025 -> (8760, 2) | fallidos: 0
Precio 2019-2025 -> (61368, 2)
Guardado: precio_bolsa_2019_2025.csv


In [15]:
# Celda 15 — Eliminar los archivos viejos (2019-2024), ya reemplazados por 2019-2025

import os  # os: para verificar/eliminar archivos del sistema

archivos_viejos = [
    "../data/precio_bolsa_2019_2024.csv",
    "../data/volumen_util_embalses_2019_2024.csv",
    "../data/aportes_hidricos_2019_2024.csv",
]

for archivo in archivos_viejos:
    if os.path.exists(archivo):
        os.remove(archivo)
        print(f"Eliminado: {archivo}")
    else:
        print(f"No encontrado (ya no existia): {archivo}")

# Confirmamos que ahora solo quedan los archivos 2019-2025 en data/
print("\nContenido actual de data/:")
print([f for f in os.listdir("../data") if f.endswith(".csv")])

Eliminado: ../data/precio_bolsa_2019_2024.csv
Eliminado: ../data/volumen_util_embalses_2019_2024.csv
Eliminado: ../data/aportes_hidricos_2019_2024.csv

Contenido actual de data/:
['aportes_hidricos_2019_2025.csv', 'precio_bolsa_2019_2025.csv', 'volumen_util_embalses_2019_2025.csv']


In [16]:
# Celda 16 — Buscar en el catalogo las metricas de demanda y generacion
#
# Reutilizamos la variable 'catalogo' que ya esta en memoria (la trajimos en la
# celda 4 de este mismo notebook, para buscar "embalse"). No hace falta volver
# a llamar get_collections().

mascara_demanda = catalogo.apply(lambda columna: columna.astype(str).str.contains("demanda", case=False, na=False))
print("=== Resultados para 'demanda' ===")
print(catalogo[mascara_demanda.any(axis=1)][["MetricId", "MetricName", "Entity", "Type", "MetricUnits"]])

# Buscamos "generaci" (sin acento y truncado) para capturar tanto "generación" como "Generacion"
mascara_generacion = catalogo.apply(lambda columna: columna.astype(str).str.contains("generaci", case=False, na=False))
print("\n=== Resultados para 'generación' ===")
print(catalogo[mascara_generacion.any(axis=1)][["MetricId", "MetricName", "Entity", "Type", "MetricUnits"]])

=== Resultados para 'demanda' ===
            MetricId                                         MetricName  \
0           DemaReal                           Demanda Real por Sistema   
1           DemaReal                            Demanda Real por Agente   
3           DemaCome                      Demanda Comercial por Sistema   
6       PrecBolsNaci                  Precio Bolsa Nacional por Sistema   
12          DemaCome                       Demanda Comercial por Agente   
..               ...                                                ...   
143    DDVContratada                         DDV Contratada por Recurso   
144       DemaMaxPot             Demanda Máxima de Potencia por Sistema   
182   EscDemUPMEAlto  Demanda de Energia Proyectada UPME Escenario A...   
183  EscDemUPMEMedio  Demanda de Energia Proyectada UPME Escenario M...   
184   EscDemUPMEBajo  Demanda de Energia Proyectada UPME Escenario B...   

      Entity             Type MetricUnits  
0    Sistema   Hourly

In [17]:
# Celda 17 — Buscar si existe una metrica de generacion YA desglosada
# por tipo (hidraulica/termica) a nivel Sistema, antes de asumir que
# hay que construirla nosotros desde el detalle por recurso.
#
# Uso una expresion regular simple ([eé] y [aá]) para que el buscador
# encuentre el termino tanto con tilde como sin tilde, por si el catalogo
# no es consistente en su escritura.

mascara_termica = catalogo.apply(lambda columna: columna.astype(str).str.contains("t[eé]rmic", case=False, na=False, regex=True))
print("=== Resultados para 'termica/térmica' ===")
print(catalogo[mascara_termica.any(axis=1)][["MetricId", "MetricName", "Entity", "Type", "MetricUnits"]])

mascara_hidraulica = catalogo.apply(lambda columna: columna.astype(str).str.contains("hidr[aá]ulic", case=False, na=False, regex=True))
print("\n=== Resultados para 'hidraulica/hidráulica' ===")
print(catalogo[mascara_hidraulica.any(axis=1)][["MetricId", "MetricName", "Entity", "Type", "MetricUnits"]])

=== Resultados para 'termica/térmica' ===
                MetricId                                         MetricName  \
6           PrecBolsNaci                  Precio Bolsa Nacional por Sistema   
8    ConsCombustibleMBTU               Consumo Combustible MBTU por Recurso   
9    ConsCombustibleMBTU                           Consumo Combustible MBTU   
22         ConsCombAprox  Consumo Combustible Aproximado para el Factor ...   
130              DemaSIN                    Demanda Energia SIN por Sistema   

          Entity            Type MetricUnits  
6        Sistema  HourlyEntities     COP/kWh  
8        Recurso  HourlyEntities        MBTU  
9    Combustible  HourlyEntities        MBTU  
22   RecursoComb  HourlyEntities        MBTU  
130      Sistema   DailyEntities         kWh  

=== Resultados para 'hidraulica/hidráulica' ===
             MetricId                       MetricName   Entity  \
130           DemaSIN  Demanda Energia SIN por Sistema  Sistema   
150  VoluUtilDiarM

In [18]:
# Celda 18 — Confirmar que DemaReal y Gene (Sistema) comparten la misma
# estructura horaria que el precio de bolsa, antes de generalizar la funcion.
import json
cuerpo_demanda = {"MetricId": "DemaReal", "StartDate": "2024-01-01", "EndDate": "2024-01-01", "Entity": "Sistema"}
resp_demanda = requests.post("https://servapibi.xm.com.co/hourly", json=cuerpo_demanda)
print("=== DemaReal ===")
print(json.dumps(resp_demanda.json()["Items"][0], indent=2, ensure_ascii=False)[:400])

cuerpo_generacion = {"MetricId": "Gene", "StartDate": "2024-01-01", "EndDate": "2024-01-01", "Entity": "Sistema"}
resp_generacion = requests.post("https://servapibi.xm.com.co/hourly", json=cuerpo_generacion)
print("\n=== Gene (Sistema) ===")
print(json.dumps(resp_generacion.json()["Items"][0], indent=2, ensure_ascii=False)[:400])

=== DemaReal ===
{
  "Date": "2024-01-01",
  "HourlyEntities": [
    {
      "Id": "Sistema",
      "Values": {
        "code": "Sistema",
        "Hour01": "7236893.31000",
        "Hour02": "7104164.61000",
        "Hour03": "7012446.40000",
        "Hour04": "6870115.23000",
        "Hour05": "6759771.97000",
        "Hour06": "6671795.60000",
        "Hour07": "6287595.26000",
        "Hour08": "6319646.62000"

=== Gene (Sistema) ===
{
  "Date": "2024-01-01",
  "HourlyEntities": [
    {
      "Id": "Sistema",
      "Values": {
        "code": "Sistema",
        "Hour01": "7343431.33000",
        "Hour02": "7211670.17000",
        "Hour03": "7120953.27000",
        "Hour04": "6979871.22000",
        "Hour05": "6874838.53000",
        "Hour06": "6719708.19000",
        "Hour07": "6353959.47000",
        "Hour08": "6413032.87000"


In [19]:
# Celda 19 — Funcion generica para descargar metricas HORARIAS (Sistema)
#
# DemaReal y Gene comparten la misma estructura de JSON que PrecBolsNaci
# (confirmado en la celda 18): HourlyEntities -> Values -> Hour01...Hour24.
# En vez de copiar descargar_precio_bolsa dos veces, la generalizamos para
# que reciba el MetricId y el nombre de columna como parametros.

def descargar_metrica_horaria(metric_id, fecha_inicio, fecha_fin, nombre_columna, entity="Sistema"):
    """
    Descarga una metrica HORARIA de la API de XM (endpoint /hourly).
    Maximo 31 dias por llamado.

    Parametros:
        metric_id: codigo de la metrica, ej. 'DemaReal' o 'Gene'
        fecha_inicio, fecha_fin: strings 'YYYY-MM-DD'
        nombre_columna: como se llamara la columna de valores en el DataFrame resultado
        entity: nivel de agregacion, 'Sistema' por defecto

    Devuelve:
        DataFrame de pandas en formato LARGO: columnas 'fecha_hora' y nombre_columna.
    """
    cuerpo_peticion = {
        "MetricId": metric_id,
        "StartDate": fecha_inicio,
        "EndDate": fecha_fin,
        "Entity": entity
    }

    respuesta = requests.post("https://servapibi.xm.com.co/hourly", json=cuerpo_peticion)
    respuesta.raise_for_status()
    datos = respuesta.json()

    filas = []
    for dia in datos["Items"]:
        fecha = dia["Date"]
        valores_del_dia = dia["HourlyEntities"][0]["Values"]
        for hora in range(1, 25):
            clave_hora = f"Hour{hora:02d}"
            valor = float(valores_del_dia[clave_hora])
            fecha_hora = pd.Timestamp(fecha) + pd.Timedelta(hours=hora - 1)
            filas.append({"fecha_hora": fecha_hora, nombre_columna: valor})

    return pd.DataFrame(filas)


def descargar_rango_horario_robusto(metric_id, fecha_inicio_total, fecha_fin_total, nombre_columna,
                                      dias_por_bloque=30, max_reintentos=3, entity="Sistema"):
    """Descarga una metrica horaria para un rango largo, en bloques, con reintentos."""
    fecha_actual = pd.Timestamp(fecha_inicio_total)
    fecha_limite = pd.Timestamp(fecha_fin_total)
    bloques_descargados = []
    bloques_fallidos = []
    numero_bloque = 1

    while fecha_actual <= fecha_limite:
        fin_de_bloque = min(fecha_actual + pd.Timedelta(days=dias_por_bloque - 1), fecha_limite)
        inicio_str = fecha_actual.strftime("%Y-%m-%d")
        fin_str = fin_de_bloque.strftime("%Y-%m-%d")

        exito = False
        for intento in range(1, max_reintentos + 1):
            try:
                df_bloque = descargar_metrica_horaria(metric_id, inicio_str, fin_str, nombre_columna, entity)
                bloques_descargados.append(df_bloque)
                print(f"[{metric_id}] Bloque {numero_bloque} OK: {inicio_str} a {fin_str} ({len(df_bloque)} filas)")
                exito = True
                break
            except Exception as error:
                print(f"  [{metric_id}] Bloque {numero_bloque} intento {intento} fallo: {error}")
                time.sleep(intento * 3)

        if not exito:
            bloques_fallidos.append((inicio_str, fin_str))
            print(f"  [{metric_id}] Bloque {numero_bloque} DESCARTADO tras {max_reintentos} intentos")

        fecha_actual = fin_de_bloque + pd.Timedelta(days=1)
        numero_bloque += 1
        time.sleep(1)

    if bloques_descargados:
        df_completo = pd.concat(bloques_descargados, ignore_index=True)
        df_completo = df_completo.sort_values("fecha_hora").drop_duplicates("fecha_hora").reset_index(drop=True)
    else:
        df_completo = pd.DataFrame()

    return df_completo, bloques_fallidos

print("Funciones genericas de descarga horaria listas")

Funciones genericas de descarga horaria listas


In [20]:
# Celda 20 — Mini-prueba: confirmar datos desde 2019 para demanda y generacion
df_test_dem = descargar_metrica_horaria("DemaReal", "2019-01-01", "2019-01-02", "demanda_kwh")
print("Demanda 2019 ->", df_test_dem.shape)
print(df_test_dem.head(3))

df_test_gen = descargar_metrica_horaria("Gene", "2019-01-01", "2019-01-02", "generacion_kwh")
print("\nGeneracion 2019 ->", df_test_gen.shape)
print(df_test_gen.head(3))

Demanda 2019 -> (48, 2)
           fecha_hora  demanda_kwh
0 2019-01-01 00:00:00   6276771.07
1 2019-01-01 01:00:00   6086853.43
2 2019-01-01 02:00:00   5896211.05

Generacion 2019 -> (48, 2)
           fecha_hora  generacion_kwh
0 2019-01-01 00:00:00      6235333.96
1 2019-01-01 01:00:00      6059405.81
2 2019-01-01 02:00:00      5870845.72


In [21]:
# Celda 21 — Descarga COMPLETA 2019-2025 de demanda y generacion, y guardado en disco

# --- Demanda real del sistema ---
df_demanda, fallos_demanda = descargar_rango_horario_robusto("DemaReal", "2019-01-01", "2025-12-31", "demanda_kwh")

print("\n===== RESUMEN DEMANDA =====")
print("Total de filas:", df_demanda.shape[0])
print("Rango de fechas:", df_demanda["fecha_hora"].min(), "a", df_demanda["fecha_hora"].max())
print("Bloques fallidos:", len(fallos_demanda))

df_demanda.to_csv("../data/demanda_real_2019_2025.csv", index=False)
print("Guardado en: ../data/demanda_real_2019_2025.csv")

# --- Generacion total del sistema ---
df_generacion, fallos_generacion = descargar_rango_horario_robusto("Gene", "2019-01-01", "2025-12-31", "generacion_kwh")

print("\n===== RESUMEN GENERACION =====")
print("Total de filas:", df_generacion.shape[0])
print("Rango de fechas:", df_generacion["fecha_hora"].min(), "a", df_generacion["fecha_hora"].max())
print("Bloques fallidos:", len(fallos_generacion))

df_generacion.to_csv("../data/generacion_total_2019_2025.csv", index=False)
print("Guardado en: ../data/generacion_total_2019_2025.csv")

[DemaReal] Bloque 1 OK: 2019-01-01 a 2019-01-30 (720 filas)
[DemaReal] Bloque 2 OK: 2019-01-31 a 2019-03-01 (720 filas)
[DemaReal] Bloque 3 OK: 2019-03-02 a 2019-03-31 (720 filas)
[DemaReal] Bloque 4 OK: 2019-04-01 a 2019-04-30 (720 filas)
[DemaReal] Bloque 5 OK: 2019-05-01 a 2019-05-30 (720 filas)
[DemaReal] Bloque 6 OK: 2019-05-31 a 2019-06-29 (720 filas)
[DemaReal] Bloque 7 OK: 2019-06-30 a 2019-07-29 (720 filas)
[DemaReal] Bloque 8 OK: 2019-07-30 a 2019-08-28 (720 filas)
[DemaReal] Bloque 9 OK: 2019-08-29 a 2019-09-27 (720 filas)
[DemaReal] Bloque 10 OK: 2019-09-28 a 2019-10-27 (720 filas)
[DemaReal] Bloque 11 OK: 2019-10-28 a 2019-11-26 (720 filas)
[DemaReal] Bloque 12 OK: 2019-11-27 a 2019-12-26 (720 filas)
[DemaReal] Bloque 13 OK: 2019-12-27 a 2020-01-25 (720 filas)
[DemaReal] Bloque 14 OK: 2020-01-26 a 2020-02-24 (720 filas)
[DemaReal] Bloque 15 OK: 2020-02-25 a 2020-03-25 (720 filas)
[DemaReal] Bloque 16 OK: 2020-03-26 a 2020-04-24 (720 filas)
[DemaReal] Bloque 17 OK: 2020-04-